# Achilles LoRA Stage 2 — Bulut GPU Egitimi (UCRETSIZ T4)

**Base:** `{BASE_MODEL}`  (Ollama `qwen3:4b-instruct-2507` ile BIREBIR ayni checkpoint)

**Girdi:** `{HF_DATASET_REPO}` icindeki JSONL — her satir `{"messages":[{system},{user},{assistant}]}`

**Cikti:** `{ADAPTER_NAME}.Q4_K_M.gguf` + `Modelfile`  -> kullanici lokalde `ollama create achilles -f Modelfile`

## Platform secimi
- **Kaggle (onerilen):** Settings > Accelerator = `GPU T4 x2`, Internet = ON. Haftalik 30 saat kota, cikti kalici (Output sekmesi).
- **Colab free:** Runtime > Change runtime type > T4 GPU. Oturum kopabilir; cikti `files.download` / Drive'a alinmali.

> CLAUDE.md kural 2: GGUF uretimi 'egitim bitti' demektir, 'calisiyor' demek DEGIL. Indikten sonra `ollama run` + eval gate sart.

In [ ]:
# === HUCRE 1: Tek-GPU pinleme (T4x2'de tek T4'e sabitle) ===
# KRITIK: CUDA_VISIBLE_DEVICES torch/unsloth import'undan ONCE set edilmeli.
# Import sonrasi etkisiz kalir; T4x2'de unsloth acik-kaynak tek-GPU yolu iki cihaz
# gorup hata/yavaslama uretir. Bu hucre import iceren her hucreden ONCE calismali.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"      # yalniz 1. T4 (16GB) — 4B+QLoRA sigar
os.environ["UNSLOTH_RETURN_LOGITS"] = "1"
print("CUDA_VISIBLE_DEVICES =", os.environ["CUDA_VISIBLE_DEVICES"])

In [ ]:
# === HUCRE 2: Bagimliliklar ===
# unsloth GGUF q4_k_m 'GGGG'/NaN bug'i (Issue #2860) v2025.6.12'de vardi, sonrasi DUZELTILDI.
# En guncel unsloth + DOGRULANMIS trl/transformers cifti kullaniyoruz.
!pip install -q --upgrade pip
!pip install -q unsloth unsloth_zoo
# Qwen3-2507 notebook'unda dogrulanmis kombinasyon:
!pip install -q --no-deps trl==0.22.2 transformers==4.56.2

import unsloth, transformers, trl, torch
print("unsloth     :", unsloth.__version__)        # >= 2025.7.x olmali (NaN fix sonrasi)
print("transformers:", transformers.__version__, "| trl:", trl.__version__)
print("torch       :", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), "GPU yok — Accelerator = GPU T4 sec."

In [ ]:
# === HUCRE 3: Yapilandirma (placeholder degerler enjekte edilir) ===
# DIKKAT: BASE_MODEL Ollama 'qwen3:4b-instruct-2507' ile AYNI checkpoint olmali;
# ciplak 'qwen3:4b' tag'i hybrid-thinking 2504 modelidir — adapter ona oturmaz.
BASE_MODEL      = "{BASE_MODEL}"           # = Qwen/Qwen3-4B-Instruct-2507
ADAPTER_NAME    = "{ADAPTER_NAME}"         # cikti klasor/model adi, orn achilles_lora_cloud
HF_DATASET_REPO = "{HF_DATASET_REPO}"      # private HF dataset repo, orn KULLANICI/achilles-lora-sft
DATA_FILE       = "lora_sft.jsonl"          # repo icindeki JSONL dosya adi

MAX_SEQ_LEN     = {MAX_SEQ_LEN}            # T4 guvenli 2048; kisa veride 1024
LORA_R          = {LORA_R}                # 4B icin 16-32; kucuk veride 16
LORA_ALPHA      = {LORA_ALPHA}            # alpha; tipik 2*r (kucuk veride stabil)
LORA_DROPOUT    = {LORA_DROPOUT}          # 0 -> unsloth fast-path; >0 regularizasyon
USE_RSLORA      = {USE_RSLORA}            # rank-stabilized scaling (alpha/sqrt(r))
NEFTUNE_ALPHA   = {NEFTUNE_ALPHA}         # NEFTune embedding gurultu (0=kapali, tipik 5)
WEIGHT_DECAY    = {WEIGHT_DECAY}          # AdamW weight decay (regularizasyon)
WARMUP_RATIO    = {WARMUP_RATIO}          # lr warmup orani
LEARNING_RATE   = {LEARNING_RATE}         # LoRA tipik 2e-4
NUM_EPOCHS      = {NUM_EPOCHS}            # ~1000+ ornek icin 2-3
SEED            = 42                        # CLAUDE.md kural 6: determinizm

import random, numpy as np
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("Config OK:", BASE_MODEL, "| r=", LORA_R, "| epochs=", NUM_EPOCHS, "| max_seq=", MAX_SEQ_LEN)

In [ ]:
# === HUCRE 4: HF token (OPSIYONEL — Kaggle'a veriyi DOGRUDAN yuklediysen GEREKMEZ) ===
# Veriyi Kaggle Dataset olarak yuklediysen bu hucre HF_TOKEN=None birakir ve gecer.
# HF private repo kullaniyorsan: Kaggle Add-ons > Secrets > name='HF_TOKEN'.
import os
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    print('Platform: Kaggle (HF_TOKEN bulundu)')
except Exception:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
        print('Platform: Colab (HF_TOKEN bulundu)')
    except Exception:
        HF_TOKEN = os.environ.get('HF_TOKEN')
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print('HF login OK')
else:
    print('HF_TOKEN yok -> veri Kaggle Dataset (/kaggle/input) icinden okunacak.')

In [ ]:
# === HUCRE 5: Egitim verisini bul (HF varsa oradan; yoksa Kaggle Dataset'ten) ===
import glob
local_jsonl = None
if HF_TOKEN:
    try:
        from huggingface_hub import hf_hub_download
        local_jsonl = hf_hub_download(repo_id=HF_DATASET_REPO, filename=DATA_FILE,
                                      repo_type='dataset', token=HF_TOKEN)
        print('HF dataset indirildi:', local_jsonl)
    except Exception as e:
        print('HF indirme basarisiz, Kaggle input deneniyor:', e)
if not local_jsonl:
    cands = sorted(glob.glob('/kaggle/input/**/*.jsonl', recursive=True)) + \
            sorted(glob.glob('/content/**/*.jsonl', recursive=True))
    assert cands, 'JSONL bulunamadi — Kaggle sag panel > Add Input ile dataset ekle.'
    local_jsonl = cands[0]
    print('Kaggle/lokal JSONL:', local_jsonl)
print('Veri dosyasi:', local_jsonl)

In [ ]:
# === HUCRE 6: Base model + tokenizer (4-bit QLoRA, T4'e sigar) ===
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = BASE_MODEL,        # Qwen3-4B-Instruct-2507 = Ollama qwen3:4b-instruct-2507
    max_seq_length  = MAX_SEQ_LEN,
    dtype           = None,              # None -> T4'te otomatik float16 (T4 bf16 DESTEKLEMEZ)
    load_in_4bit    = True,              # QLoRA — 16GB'a sigmasi icin sart
    load_in_8bit    = False,             # 8-bit Qwen3-2507'de bilinen bug (Issue #3501)
    full_finetuning = False,
)
print("Base yuklendi:", BASE_MODEL)

In [ ]:
# === HUCRE 7: LoRA adapter (BUG 1 fix — 7 modul acikca, lm_head/embed YOK) ===
# KRITIK: Qwen3 tied-embeddings. lm_head/embed_tokens EKLERSEN GGUF'ta deltalar
# sessizce atlanir ve adapter bozulur. Yalniz attention+MLP projeksiyonlari.
model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],   # 7 modul = full LoRA
    lora_alpha = LORA_ALPHA,              # alpha (config'ten; tipik 2*r)
    lora_dropout = LORA_DROPOUT,          # 0 -> unsloth fast-path; >0 regularizasyon
    bias = "none",                        # unsloth optimize tek desteklenen deger
    use_gradient_checkpointing = "unsloth",  # VRAM ~%30 duser — T4 icin kritik
    random_state = SEED,                  # determinizm
    use_rslora = USE_RSLORA,
    loftq_config = None,
)
model.print_trainable_parameters()

In [ ]:
# === HUCRE 8: Veri yukle (BUG 2 fix — {messages} tuket) + chat template (BUG 3 fix) ===
import json
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

# Qwen3-Instruct-2507 ChatML sablonu (Ollama'daki <|im_start|>... ile ayni).
# 'qwen3-instruct' (Thinking icin duz 'qwen3' DEGIL) — 2507 non-thinking, <think> YOK.
tokenizer = get_chat_template(tokenizer, chat_template="qwen3-instruct")

# BUG 2 fix: Achilles dataset_builder ciktisi {'messages':[...]} — dogrudan tuket.
rows = []
with open(local_jsonl, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        msgs = obj.get("messages") if isinstance(obj, dict) else None
        if not msgs and isinstance(obj, dict):
            # geriye-donuk tolerans: {system/user/assistant} ya da {text}
            if any(k in obj for k in ("system", "user", "assistant")):
                msgs = [{"role": r, "content": obj[r]}
                        for r in ("system", "user", "assistant") if obj.get(r)]
            elif obj.get("text"):
                msgs = [{"role": "user", "content": obj["text"]}]
        msgs = [m for m in (msgs or []) if m.get("content")]
        if msgs:
            rows.append({"messages": msgs})
print("Toplam ornek:", len(rows))
assert len(rows) >= 50, "Cok az ornek — once 'achilles synth-qa' ile >=1000 uret (Stage 1)."

# Determinist train/valid split
random.Random(SEED).shuffle(rows)
n_valid = max(1, int(len(rows) * 0.05))
valid_rows, train_rows = rows[:n_valid], rows[n_valid:]

# BUG 3 fix: GERCEK chat template ile render (uydurma <|user|> DEGIL).
# add_generation_prompt=False -> egitimde tam diyalog, sonda bos assistant prompt YOK.
def render(example):
    return {"text": tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False)}

train_ds = Dataset.from_list(train_rows).map(render, remove_columns=["messages"])
valid_ds = Dataset.from_list(valid_rows).map(render, remove_columns=["messages"])
print(f"train={len(train_ds)}  valid={len(valid_ds)}")
print("--- ORNEK RENDER (Turkce + <|im_start|> dogrula, <think> OLMAMALI) ---")
print(train_ds[0]["text"][:700])

In [ ]:
# === HUCRE 9: Trainer (BUG 4 fix — dinamik padding, max_length DEGIL) ===
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = valid_ds,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ_LEN,            # truncate; padding DINAMIK (max_length degil)
        per_device_train_batch_size = 2,        # T4 16GB; OOM olursa 1
        gradient_accumulation_steps = 8,        # 2*8 = efektif batch 16 (tek GPU)
        warmup_ratio = WARMUP_RATIO,
        num_train_epochs = NUM_EPOCHS,
        learning_rate = LEARNING_RATE,
        logging_steps = 5,
        optim = "adamw_8bit",                    # 8-bit optimizer -> VRAM tasarrufu
        weight_decay = WEIGHT_DECAY,
        neftune_noise_alpha = (NEFTUNE_ALPHA or None),  # 0.0 -> kapali
        lr_scheduler_type = "cosine",
        fp16 = True, bf16 = False,               # T4: fp16 (bf16 NaN riski)
        packing = False,                         # chat verisi + maskeleme hassasiyeti
        eval_strategy = "epoch",
        save_strategy = "epoch", save_total_limit = 1,
        seed = SEED,
        output_dir = ADAPTER_NAME,
        report_to = "none",
    ),
)

In [ ]:
# === HUCRE 10: Sadece assistant cevaplarina loss (system/user maskele) ===
# instruction/response_part Qwen3 ChatML token'lariyla BIREBIR eslesmeli (sondaki \n dahil),
# yoksa maskeleme hicbir sey bulamaz ve model hic ogrenmez (Issue #2771, #1017).
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)

# Dogrulama: loss uygulanan (label != -100) token sayisi 0 OLMAMALI.
import numpy as np
labels = trainer.train_dataset[0]["labels"]
n_loss = int(np.sum(np.array(labels) != -100))
print("loss uygulanan token sayisi:", n_loss)
assert n_loss > 0, "Maskeleme bos — part string'leri template ile uyusmuyor (qwen3-instruct kontrol et)."

In [ ]:
# === HUCRE 11: EGIT ===
trainer_stats = trainer.train()
print(trainer_stats)

# Loss egrisini Achilles web grafigi formatinda kaydet (reports/training/ okur)
import json
curve = [{"step": int(e.get("step", i + 1)),
          "train_loss": round(float(e["loss"]), 4),
          "val_loss": (round(float(e["eval_loss"]), 4) if "eval_loss" in e else None)}
         for i, e in enumerate(trainer.state.log_history) if "loss" in e]
json.dump({"adapter_name": ADAPTER_NAME, "base_model": BASE_MODEL,
           "total_iters": trainer.state.global_step, "curve": curve},
          open(f"{ADAPTER_NAME}_loss.json", "w"), ensure_ascii=False, indent=2)
print("Loss egrisi:", f"{ADAPTER_NAME}_loss.json")

In [ ]:
# === HUCRE 12: GGUF Q4_K_M + Modelfile (BUG 5 fix — YOL A: unsloth kisa yol) ===
# Guncel unsloth'ta (v2025.7+) Qwen3 q4_k_m bug'i duzeltildi. Bu cagri merge+convert+quantize
# yapar VE Ollama Modelfile'ini otomatik uretir (dogru TEMPLATE + stop token'lar).
model.save_pretrained_gguf(
    ADAPTER_NAME + "_gguf",
    tokenizer,
    quantization_method = "q4_k_m",   # kucuk harf, alt cizgi
)

import glob, shutil, os
gguf = (sorted(glob.glob(f"{ADAPTER_NAME}_gguf/*Q4_K_M*.gguf")) or
        sorted(glob.glob(f"{ADAPTER_NAME}_gguf/*q4_k_m*.gguf")) or
        sorted(glob.glob(f"{ADAPTER_NAME}_gguf/*.gguf")))
assert gguf, "GGUF uretilmedi — HUCRE 12B fallback'ini calistir."
shutil.copy(gguf[0], "achilles-Q4_K_M.gguf")
print("GGUF:", round(os.path.getsize("achilles-Q4_K_M.gguf") / 1e9, 2), "GB")

# unsloth'un urettigi Modelfile (TEMPLATE + stop token'lar dogru gelir)
try:
    print("--- unsloth Modelfile ---")
    print(tokenizer._ollama_modelfile)
except Exception as e:
    print("Modelfile otomatik gelmedi, HUCRE 13'teki elle Modelfile'i kullan:", e)

In [ ]:
# === HUCRE 12B (FALLBACK): garbage GGUF ('GGGG'/NaN) cikarsa bunu calistir ===
# Issue #2860 workaround: once 16-bit merge -> llama.cpp f16 -> manuel Q4_K_M quantize.
# YOL A sorunsuzsa bu hucreyi ATLA.
RUN_FALLBACK = False   # YOL A garbage uretirse True yap
if RUN_FALLBACK:
    model.save_pretrained_merged("achilles_merged_16bit", tokenizer, save_method="merged_16bit")
    !git clone --depth 1 https://github.com/ggerganov/llama.cpp
    !pip install -q -r llama.cpp/requirements.txt
    !cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF >/dev/null 2>&1
    !cmake --build llama.cpp/build --target llama-quantize --config Release -j 2 >/dev/null 2>&1
    # 4-bit DEGIL, merged 16-bit klasoru convert edilir (NaN riski dusuk)
    !python llama.cpp/convert_hf_to_gguf.py achilles_merged_16bit --outfile achilles-f16.gguf --outtype f16
    import glob as _g
    _QBIN = (_g.glob('llama.cpp/build/bin/llama-quantize') or ['llama.cpp/build/bin/llama-quantize'])[0]
    !{_QBIN} achilles-f16.gguf achilles-Q4_K_M.gguf Q4_K_M
    print("Fallback hazir: achilles-Q4_K_M.gguf (Modelfile'i HUCRE 13 ile elle yaz)")
else:
    print("Fallback atlandi (YOL A calisti).")

In [ ]:
# === HUCRE 13: Ollama Modelfile (fallback durumunda elle; YOL A'da yedek) ===
# YOL A'da unsloth Modelfile'i kullanmak yeterli; bu hucre fallback icin garanti yol.
# TEMPLATE egitimdeki apply_chat_template ('qwen3-instruct') ile BIREBIR ayni ChatML.
modelfile = '''FROM ./achilles-Q4_K_M.gguf

TEMPLATE """{{ if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{ if .Prompt }}<|im_start|>user
{{ .Prompt }}<|im_end|>
{{ end }}<|im_start|>assistant
{{ .Response }}<|im_end|>
"""

SYSTEM """Sen Achilles yerel AI asistanisin. Matematik, fizik, istatistik, felsefe, trading ve AI sistem tasarimi konularinda adim adim, kaynak temelli, kontrollu ve belirsizligi dogru ifade eden cevaplar ver. RAG baglami varsa kullan. Kaynak yoksa emin gorunme."""

PARAMETER stop "<|im_start|>"
PARAMETER stop "<|im_end|>"
PARAMETER temperature 0.7
PARAMETER top_p 0.8
PARAMETER top_k 20
PARAMETER min_p 0.0
PARAMETER num_ctx 8192
'''
with open("Modelfile", "w", encoding="utf-8") as f:
    f.write(modelfile)
print(modelfile)

In [ ]:
# === HUCRE 14: Ciktilari indir ===
# Kaggle: cikti /kaggle/working/ altinda -> sag panel 'Output' sekmesinden indir.
#         (Notebook'u 'Save & Run All (Commit)' yap ki Output kalici olsun.)
# Colab : oturum silinir -> files.download veya Drive'a kopyala.
import shutil, os
for f in ["achilles-Q4_K_M.gguf", "Modelfile", f"{ADAPTER_NAME}_loss.json"]:
    if os.path.exists(f):
        print("Cikti:", f, "(", round(os.path.getsize(f) / 1e6, 1), "MB )")

try:
    from google.colab import files
    files.download("achilles-Q4_K_M.gguf")
    files.download("Modelfile")
    files.download(f"{ADAPTER_NAME}_loss.json")
except Exception:
    print("Kaggle: yukaridaki dosyalari 'Output' sekmesinden indir.")

## LOKAL MAKINE (Windows / PowerShell) — Ollama'ya yukle

Indirilen `achilles-Q4_K_M.gguf` + `Modelfile` AYNI klasorde olsun.

```powershell
# FROM yolu goreli (./achilles-Q4_K_M.gguf) — komutu o klasorden calistir
ollama create achilles -f Modelfile
ollama run achilles "Sharpe oranini ve sinirlamalarini acikla."   # duman testi
ollama list                                                       # 'achilles' gorunmeli
ollama show achilles --modelfile                                  # TEMPLATE/stop dogrula
```

> `<|im_end|>` stop token'i eksikse model durmaz, sonsuz uretir. Modelfile'da SART.

## Eval gate (CLAUDE.md kural 2 — test edilmeden 'calisiyor' deme)
```powershell
$env:ACHILLES_LLM_MODEL="achilles"; uv run achilles evaluate evals\discipline_core.jsonl --adapter-version achilles
# Hedef: score=1.0, total_flags=0 (guaranteed_profit / ignores_costs bayraklari dusmemeli)
```